# Preliminary MG portfolio results

This notebook plots the **new capacity built in each single-year PyPSA portfolio case** using the team's saved `planning_study/results/portfolio_pypsa.csv`. It does not solve the model or choose generator sites. Run it from any folder inside a checkout of this repository. Results are preliminary until the stacked planning-study PRs are reviewed and integrated.

The 2035 and 2050 rows are separate model cases. The chart shows **new build (GW)**, not the full installed fleet, hourly generation, or a cumulative construction schedule.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'planning_study' / 'results' / 'portfolio_pypsa.csv').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook inside a checkout containing planning_study/results/portfolio_pypsa.csv')
RESULTS = ROOT / 'planning_study' / 'results'
FIGURES = ROOT / 'planning_study' / 'notebooks' / 'figures'
print('Repository:', ROOT)


In [ ]:
source = RESULTS / 'portfolio_pypsa.csv'
d = pd.read_csv(source)
cases = ['REFRESH_2035', 'REFRESH_2050']
required = ['case', 'year', 'status', 'condition', 'objective_mbrl', 'imports_twh', 'curtailment_twh']
techs = [('solar_central', 'Central PV'), ('solar_distributed', 'Distributed PV'), ('bio', 'Biomass'), ('small_hydro', 'Small hydro'), ('wind_site', 'Wind'), ('biogas', 'Biogas'), ('bess', 'BESS (power)')]
required += [f'cap_{key}_gw' for key, _ in techs]
missing = set(required) - set(d.columns)
if missing:
    raise ValueError(f'Missing expected portfolio columns: {sorted(missing)}')
base = d.set_index('case').loc[cases].copy()
assert base['year'].tolist() == [2035, 2050]
assert (base['status'] == 'ok').all() and (base['condition'] == 'optimal').all()
assert base[[f'cap_{key}_gw' for key, _ in techs]].notna().all().all()
base.index = [2035, 2050]
base.index.name = 'Year'
print('Input:', source.relative_to(ROOT))
display(base[['objective_mbrl', 'imports_twh', 'curtailment_twh']].round(2))


In [ ]:
capacity = pd.DataFrame({label: base[f'cap_{key}_gw'].clip(lower=0) for key, label in techs}, index=base.index)
colors = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#008300', '#6250d6']
fig, ax = plt.subplots(figsize=(10, 3.6))
left = pd.Series(0.0, index=capacity.index)
for (label, values), color in zip(capacity.items(), colors):
    ax.barh(capacity.index.astype(str), values, left=left, label=label, color=color, edgecolor='white')
    left += values
for year, total in left.items():
    ax.text(total + 0.3, str(year), f'{total:.1f} GW', va='center')
ax.set_xlim(0, max(left) * 1.16)
ax.invert_yaxis()
ax.set_xlabel('New capacity in each model case (GW)')
ax.set_title('Preliminary MG portfolio: new capacity by technology')
ax.legend(ncol=4, frameon=False, loc='lower left', bbox_to_anchor=(0, 1.01))
ax.grid(axis='x', alpha=0.25)
fig.tight_layout()
FIGURES.mkdir(exist_ok=True)
out = FIGURES / 'preliminary_portfolio_mix.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', out.relative_to(ROOT))
display(capacity.round(3))


## How to describe this figure

- These are **model-selected new capacities**, including BESS power capacity. BESS energy capacity depends on its modeled duration.
- The CSV is the saved PyPSA reproduction of the team's portfolio LP; it is not a measured forecast. The two years have different load and policy assumptions, and their cost figures use different model horizons, so compare the capacity mix rather than treating the objective values as directly comparable annual costs.
- The transmission analysis tests how a fixed portfolio interacts with the grid. It does not establish site-level resource potential or final connection feasibility. Generation siting is left for later study.
